# Cost Attribution on Databricks — Walkthrough

**Goal:** Attribute Databricks DBU cost three ways using GA system tables.

| View | Question it answers | Source tables |
|------|---------------------|---------------|
| 👤 Per-user | "Which user / service principal is driving cost?" | `system.billing.usage` + `system.billing.list_prices` |
| 🔍 Per-query | "Approximately how much did each SQL query cost?" | `system.query.history` × `system.billing.usage` |
| 📦 Per-table / view / MV | "Which UC table or MV is most expensive to maintain?" | `system.billing.usage.usage_metadata.uc_table_*` |

The companion AI/BI dashboard uses these exact queries.

## Step 0 — The system tables you'll use

| Table | What's in it | Grain |
|-------|--------------|-------|
| `system.billing.usage` | Every billable DBU row from every workload | One row per (workspace, sku, hour, compute resource) |
| `system.billing.list_prices` | Time-versioned $ price per SKU per cloud | One row per (sku, cloud, validity window) |
| `system.query.history` | Every SQL statement executed on a warehouse | One row per `statement_id` |

The `usage_metadata` and `identity_metadata` structs on `system.billing.usage` are the **dimensions** we use to attribute cost. Inspect them first.

In [0]:
DESCRIBE system.billing.usage;

In [0]:
-- The two structs that drive attribution
SELECT
  identity_metadata.run_as       AS run_as,         -- email or SP for who/what ran the workload
  identity_metadata.created_by   AS created_by,     -- creator of the resource (cluster/warehouse/job)
  identity_metadata.owned_by     AS owned_by,       -- owner of the resource
  usage_metadata.warehouse_id    AS warehouse_id,   -- DBSQL warehouse that ran the query
  usage_metadata.cluster_id      AS cluster_id,     -- All-purpose / job cluster
  usage_metadata.job_id          AS job_id,         -- Workflow job
  usage_metadata.dlt_pipeline_id AS dlt_pipeline_id,-- DLT / streaming
  usage_metadata.uc_table_catalog AS uc_catalog,
  usage_metadata.uc_table_schema  AS uc_schema,
  usage_metadata.uc_table_name    AS uc_table_name -- UC table / view / MV the DBUs hit
FROM system.billing.usage
WHERE usage_date >= current_date() - 7
LIMIT 5;

## Step 1 — Convert raw DBUs into dollars: the `list_prices` join

`system.billing.usage.usage_quantity` is in **DBUs**, not dollars. To convert, join `system.billing.list_prices`. Two design choices:

1. **Time-correct** — match each usage row to the price valid at that timestamp (`usage_start_time BETWEEN price_start_time AND price_end_time`). Use this for historical chargeback where price changes matter.
2. **Latest-price** (used here) — pick today's active price per (sku, cloud), apply to all rows. Simpler, fast, and accurate enough for ongoing monitoring.

We use the simpler form below. Swap the `lp` CTE for the time-correct version if your prices changed materially over the window.

In [0]:
-- The reusable price CTE — every downstream query starts with this.
WITH lp AS (
  SELECT
    sku_name,
    cloud,
    pricing.default AS unit_price                  -- $ per DBU for this SKU/cloud
  FROM system.billing.list_prices
  WHERE current_timestamp() BETWEEN price_start_time
                                AND COALESCE(price_end_time, current_timestamp())
)
SELECT * FROM lp ORDER BY sku_name LIMIT 10;

## Step 2 — 👤 Per-user attribution

**Logic.** Group `system.billing.usage` by `identity_metadata.run_as` and convert DBUs → $ via the price join. `run_as` is set for SQL warehouse users, job/notebook runners, and service principals. Some rows (e.g. account-level SKUs, Apps idle time) have no `run_as` — bucket those as "(unattributed)".

**Joins:**
```
system.billing.usage  u
  ← LEFT JOIN system.billing.list_prices  ON sku_name + cloud
```

**Why LEFT JOIN?** Some new SKUs may not yet appear in list_prices — keep the row, treat unit_price as 0 so you still get the DBU count.

In [0]:
WITH lp AS (
  SELECT sku_name, cloud, pricing.default AS unit_price
  FROM system.billing.list_prices
  WHERE current_timestamp() BETWEEN price_start_time
                                AND COALESCE(price_end_time, current_timestamp())
)
SELECT
  COALESCE(u.identity_metadata.run_as, '(unattributed)') AS user_email,
  u.workspace_id,
  u.usage_date,
  u.billing_origin_product                                AS product,
  SUM(u.usage_quantity * COALESCE(lp.unit_price, 0))     AS cost_usd,
  SUM(u.usage_quantity)                                  AS dbus
FROM system.billing.usage u
LEFT JOIN lp
  ON  u.sku_name = lp.sku_name
  AND u.cloud    = lp.cloud
WHERE u.usage_date >= current_date() - 30
GROUP BY 1, 2, 3, 4
ORDER BY cost_usd DESC
LIMIT 20;

### Variations to know

- **Service-principal cost** → filter `WHERE u.identity_metadata.run_as LIKE '%@servicePrincipal%'` (or join to `system.access.workspaces_latest` for the SP display name).
- **Cost owner instead of runner** → use `identity_metadata.created_by` (creator of the cluster / warehouse / job).
- **Cost-center via custom tags** → group by `u.custom_tags['cost_center']` (or whatever tag key you adopted on clusters / warehouses / jobs).
- **Per workspace × product matrix** → drop `user_email` from GROUP BY; this is the "where is my money going" first question.

## Step 3 — 🔍 Per-query approximate cost

**Why "approximate"?** DBSQL is billed by **warehouse uptime**, not per-statement. A query running for 5 s on a warm warehouse doesn't cost the same as a 5 s query on a cold one — what you actually pay is *the warehouse-hour*, regardless of how many queries shared it.

**The trick.** Allocate each warehouse-day's $ cost to queries by their **share of total query duration on that warehouse-day**:

```
approx_cost(query) = (total_duration_ms / Σ total_duration_ms on the same warehouse-day)
                     × $ cost of that warehouse on that day
```

This is **directionally correct** for chargeback but loses precision when:

- The warehouse is mostly idle (cost is dominated by uptime, not queries) — a heavy query "wins" the allocation it didn't actually drive.
- Queries run in parallel — proportional duration overestimates a long-running query that shared compute.

**Joins:**
```
query_runtime  qr  (one row per statement, with day-total duration via window fn)
  JOIN warehouse_cost  wc  (one row per workspace × warehouse × day, $ from billing.usage)
  ON workspace_id, warehouse_id, usage_date
```

In [0]:
WITH lp AS (
  SELECT sku_name, cloud, pricing.default AS unit_price
  FROM system.billing.list_prices
  WHERE current_timestamp() BETWEEN price_start_time
                                AND COALESCE(price_end_time, current_timestamp())
),

-- a) cost per warehouse-day from billing.usage
warehouse_cost AS (
  SELECT
    u.workspace_id,
    u.usage_metadata.warehouse_id                    AS warehouse_id,
    u.usage_date,
    SUM(u.usage_quantity * COALESCE(lp.unit_price, 0)) AS day_cost_usd
  FROM system.billing.usage u
  LEFT JOIN lp
    ON u.sku_name = lp.sku_name AND u.cloud = lp.cloud
  WHERE u.usage_metadata.warehouse_id IS NOT NULL
    AND u.usage_date >= current_date() - 30
  GROUP BY 1, 2, 3
),

-- b) per-statement runtime + day-total runtime via window function
query_runtime AS (
  SELECT
    workspace_id,
    compute.warehouse_id          AS warehouse_id,
    DATE(start_time)              AS usage_date,
    statement_id,
    executed_by,
    statement_type,
    SUBSTRING(statement_text, 1, 200) AS statement_preview,
    total_duration_ms,
    SUM(total_duration_ms) OVER (
      PARTITION BY workspace_id, compute.warehouse_id, DATE(start_time)
    )                              AS day_total_ms
  FROM system.query.history
  WHERE compute.warehouse_id IS NOT NULL
    AND start_time >= current_date() - 30
)

-- c) allocate each warehouse-day's cost to its queries proportionally
SELECT
  qr.statement_id,
  qr.executed_by                              AS user_email,
  qr.usage_date,
  qr.warehouse_id,
  qr.statement_type,
  qr.statement_preview,
  qr.total_duration_ms,
  (qr.total_duration_ms * 1.0
     / NULLIF(qr.day_total_ms, 0))
   * wc.day_cost_usd                          AS approx_cost_usd
FROM query_runtime qr
JOIN warehouse_cost wc USING (workspace_id, warehouse_id, usage_date)
ORDER BY approx_cost_usd DESC
LIMIT 20;

### Variations to know

- **Allocate by `read_bytes` instead of duration** → swap `total_duration_ms` for `read_bytes` in the window + numerator. Closer to "data-shuffled" cost; biased against compute-heavy queries with small reads.
- **Filter to a dashboard or BI tool** → join on `query_source.dashboard_id` or `client_application` to attribute cost to a Tableau / Power BI / Genie surface.
- **Roll up to query "shape"** → group by `regexp_replace(statement_text, '[0-9]+', 'N')` to find the most expensive *query patterns*, not individual runs.
- **Job / notebook cost** → similar pattern, but join on `usage_metadata.job_id` / `notebook_id` from billing.usage rather than warehouse_id, and use `total_task_duration_ms` from `system.lakeflow.job_run_timeline` if you want job-grain.

## Step 4 — 📦 Per-table / view / materialized-view attribution

**Logic.** `system.billing.usage.usage_metadata.uc_table_catalog/schema/name` is **populated by the platform** for any UC-aware workload — DLT pipelines, streaming tables, materialized views, and (increasingly) SQL workloads tagged to UC tables. Group by these three fields and apply the same price join.

**What's covered:**

- DLT / Lakeflow Pipelines updating a streaming table → cost rolls up to the **target table**.
- Materialized View refresh → cost rolls up to the **MV**.
- SQL warehouse queries against UC tables — partial coverage; many SQL rows still don't carry `uc_table_*`. The query-level allocation in Step 3 fills that gap.

**Joins:**
```
system.billing.usage  u
  ← LEFT JOIN system.billing.list_prices  ON sku_name + cloud
```
No second join needed — `uc_table_*` is already in the same row.

In [0]:
WITH lp AS (
  SELECT sku_name, cloud, pricing.default AS unit_price
  FROM system.billing.list_prices
  WHERE current_timestamp() BETWEEN price_start_time
                                AND COALESCE(price_end_time, current_timestamp())
)
SELECT
  u.workspace_id,
  u.usage_metadata.uc_table_catalog AS catalog,
  u.usage_metadata.uc_table_schema  AS schema_name,
  u.usage_metadata.uc_table_name    AS table_fqn,
  u.billing_origin_product          AS product,
  u.usage_date,
  SUM(u.usage_quantity * COALESCE(lp.unit_price, 0)) AS cost_usd,
  SUM(u.usage_quantity)                              AS dbus
FROM system.billing.usage u
LEFT JOIN lp
  ON u.sku_name = lp.sku_name AND u.cloud = lp.cloud
WHERE u.usage_date >= current_date() - 30
  AND u.usage_metadata.uc_table_name IS NOT NULL
GROUP BY 1, 2, 3, 4, 5, 6
ORDER BY cost_usd DESC
LIMIT 20;

### "Cost of an MV refresh" — direct answer
To answer **"What does materialized view `cat.sch.my_mv` cost me per day?"**:

In [0]:
WITH lp AS (
  SELECT sku_name, cloud, pricing.default AS unit_price
  FROM system.billing.list_prices
  WHERE current_timestamp() BETWEEN price_start_time
                                AND COALESCE(price_end_time, current_timestamp())
)
SELECT
  u.usage_date,
  CONCAT_WS('.',
    u.usage_metadata.uc_table_catalog,
    u.usage_metadata.uc_table_schema,
    u.usage_metadata.uc_table_name)                  AS table_fqn,
  SUM(u.usage_quantity * COALESCE(lp.unit_price, 0)) AS cost_usd,
  SUM(u.usage_quantity)                              AS dbus
FROM system.billing.usage u
LEFT JOIN lp ON u.sku_name = lp.sku_name AND u.cloud = lp.cloud
WHERE u.usage_date >= current_date() - 30
  AND u.usage_metadata.uc_table_catalog IS NOT NULL
  -- Replace these three placeholders with the actual MV you care about:
  -- AND u.usage_metadata.uc_table_catalog = 'main'
  -- AND u.usage_metadata.uc_table_schema  = 'analytics'
  -- AND u.usage_metadata.uc_table_name    = 'daily_revenue_mv'
GROUP BY 1, 2
ORDER BY 1 DESC, cost_usd DESC
LIMIT 50;

## Step 5 — How the dashboard wires this together

The companion **Scapia — Cost Attribution (User · Query · Table)** AI/BI dashboard has one dataset per query above, plus filters on workspace / date / product / catalog so users can flip between **per-workspace** and **account-wide** views.

| Page | Dataset | Visuals |
|------|---------|---------|
| 👤 Per-User | `per_user` (Step 2) | KPIs, top users, daily trend by product, breakdown table |
| 🔍 Per-Query | `per_query` (Step 3) | KPIs, top users, cost by statement_type, raw-query table |
| 📦 Per-Table / MV | `per_table` (Step 4) | KPIs, top tables, daily trend, cost by catalog, table breakdown |